# Statistical validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/05_statistical_validation.ipynb)

**Answers:** Editor comments 4 and 6
**Estimated runtime:** 2 h quick / 20 h full · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Confidence intervals, permutation nulls, feature-selection stability and effect sizes.

**Two permutation nulls, deliberately distinct.** Unrestricted permutation is the
leakage test and must give AUC ~ 0.50. Within-school permutation preserves each
school's class composition, so its expected value is above chance and is *estimated*.
Conflating them produces a false leakage alarm.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
from vlpso_xai.evaluation.metrics import cluster_bootstrap_ci
from vlpso_xai.evaluation.permutation import (assert_permutation_null_is_chance,
                                              decompose_performance)
from vlpso_xai.evaluation.stability import stability_table, selection_frequency
from vlpso_xai.evaluation.effect_size import contrast_table
import glob

preds = pd.concat([pd.read_parquet(p) for p in
                   glob.glob(str(cfg.paths.checkpoints / "*_preds.parquet"))],
                  ignore_index=True)
for task, sub in preds.groupby("task"):
    ci = cluster_bootstrap_ci(sub.y_true.to_numpy(), sub.y_score.to_numpy(),
                              sub.group.to_numpy(), metric="auc",
                              n_resamples=cfg.section("statistics","bootstrap","n_resamples"))
    print(f"{task:16s} AUC {ci['estimate']:.4f}  95% CI [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")

In [ ]:
# --- Feature-selection stability ---------------------------------------
sel_long = sel[["task","method","rep","fold","selected"]]
st = stability_table(sel_long.rename(columns={"selected":"selected"}),
                     all_features=list(X.columns))
display(st)

In [ ]:
# --- Paired contrasts with multiplicity correction ---------------------
lf = sel.rename(columns={"fold":"outer_fold","rep":"repeat"})
ct = contrast_table(lf, reference="vlpso", metric="auc",
                    n_train=int(sel.n_train.mean()), n_test=int(sel.n_test.mean()),
                    correction=cfg.section("statistics","multiplicity","method"))
display(ct[["method_b","mean_difference","cohens_d_paired","magnitude","p_value","p_adjusted"]])